# The held-out test, and the justification

This chapter scores the test members named in chapter 2 with the register's final incumbent, reads that score against no skill, and
writes the justification the forecaster would act on. **It is not a blind test**: during development (2026-10-20) these same 20 test
members were scored with two candidate final models — logistic at the base-rate threshold ("precision − base rate +0.005, 95% interval
over test members [-0.020, +0.028] → NOT distinguishable from no skill") and boosting at the base-rate threshold, the result below. Read its edge as provisional; a fresh test would need data no choice was made on; chapter 7 uses the same members' SSP3-7.0 winters.


## Overview

1. Rebuild the register of chapter 5 and take its final incumbent.
2. Run the held-out test, with a counter that makes a re-run visible.
3. Read the result against no skill, and count its false alarms against its hits.
4. Justify the decision, name the result's shape, and state its limits.

Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Chapter 5](05-refinement-trials.ipynb) | Helpful | the register and its final incumbent |
| [Chapter 4](04-baselines-and-the-ladder.ipynb) | Helpful | precision, recall and AP against the base rate |
| Bootstrap intervals | Helpful | resampling with replacement |

- **Time to learn**: about 25 minutes; rebuilding the register takes about a minute.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed cache.


## Imports


In [ ]:
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lensml import (
    FEATURES,
    GOLD_CMAP,
    RNG_SEED,
    RUNG0,
    cite,
    credit_own,
    decile_edges,
    ladder,
    rebuild_register,
    setup,
    wet_by_bin,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

In [ ]:
S = setup()  # the cache, the test members, the transforms (chapters 1-2)
train, test = S.train, S.test
X_tr, y_tr = train[FEATURES].values, train["wet"].values
g_tr = train["member_id"].values
print(
    f"train: {S.n_train_members} members / {len(train)} winters   "
    f"test: {test.member_id.nunique()} members / {len(test)} winters   "
    f"N_FOLDS = {S.n_folds}"
)

## The final incumbent

The register is rebuilt here exactly as chapter 5 left it: the same trials, in the same order, from the same seeds. If you added a trial
in chapter 5, copy the same `YOUR_TRIALS` here. The final incumbent was chosen after its validation score had been seen (chapter 5),
and these test members had already scored it during development (above).


In [ ]:
YOUR_TRIALS = []  # the same list as chapter 5's Your-trial cell
reg = rebuild_register(S, YOUR_TRIALS)
print(f"incumbent after {len(reg.rows)} trials: {reg.incumbent_name}")
reg.bonferroni_line()
cols = [
    "trial",
    "candidate",
    "incumbent",
    "recall_gap",
    "called_wet_cand",
    "called_wet_inc",
    "recall_p",
    "precision_gap",
    "precision_p",
    "verdict",
]
reg.table()[cols]

## The test

The final model is fit once on **all** training members and scored on the test members. `FINAL` ships `True`, so every build reruns
and shows this test of the shipped register; re-running unchanged re-reads the same score. Change anything after reading it — a trial, a threshold, an input — and the test members have become a
validation set, to be reported as such; set `FINAL = False` while adding trials. The run counter cannot see a restart; the discipline is yours.

The cell prints the wet-call share beside every recall, the false alarms and hits, and precision and AP minus the base rate of the same
test winters, each with a 95% interval from resampling whole **test members**. That interval covers only which test members were drawn —
not the choice of model, the training members, or chapter 5's selection. A printed bound of `±0.000` reads *NOT distinguishable*.


In [ ]:
FINAL = True  # the shipped register is final; set False while adding trials

In [ ]:
if not FINAL:  # nothing reads the test labels
    print(
        f"test not run: set FINAL = True after your last trial (register so far: "
        f"{len(reg.rows)} tested rows, incumbent {reg.incumbent_name})"
    )
else:
    TEST_RUNS = (
        globals().get("TEST_RUNS", 0) + 1
    )  # a second run in one session is a second test
    X_te, y_te = test[FEATURES].values, test["wet"].values
    final = reg.incumbent_make(RNG_SEED).fit(X_tr, y_tr)
    yhat_te = final.predict(X_te)
    cm_te = confusion_matrix(y_te, yhat_te, labels=[0, 1])
    score_te = final.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = (int(v) for v in cm_te.ravel())

    print(
        f"test run #{TEST_RUNS}"
        + ("" if TEST_RUNS == 1 else "   — NOT a test: report run #1")
    )
    print(
        f"final model: {reg.incumbent_name}   test members: {test.member_id.nunique()}   "
        f"test winters: {len(y_te)}   base rate: {y_te.mean():.3f}"
    )
    acc_final = accuracy_score(y_te, yhat_te)
    print(
        f"precision (wet) = {precision_score(y_te, yhat_te, zero_division=0):.3f}   "
        f"recall (wet) = {recall_score(y_te, yhat_te, zero_division=0):.3f}   "
        f"wet-call share = {yhat_te.mean():.3f}   accuracy = {acc_final:.3f}   "
        f"AP = {average_precision_score(y_te, score_te):.3f}"
    )
    print(
        classification_report(
            y_te, yhat_te, target_names=["dry", "wet"], zero_division=0
        )
    )
    print(
        "test confusion matrix (rows = true, columns = predicted); the figure below "
        "draws the same matrix:"
    )
    print(
        pd.DataFrame(
            cm_te,
            index=["true dry", "true wet"],
            columns=["pred dry", "pred wet"],
        )
    )
    print(
        f"false alarms (true dry, called wet) {fp} · hits (true wet, called wet) {tp} · "
        f"false alarms per hit {fp / tp:.2f} · misses {fn} · correct rejections {tn}"
    )
    print(
        f"break-even: leaning wet on these calls pays only if a false alarm "
        f"costs less than hits / false alarms = {tp / fp:.2f} of a hit's value"
    )

    acc = {}
    # the two bars and the physical prior, on the same test members
    for name in RUNG0:
        yb = ladder(RNG_SEED)[name].fit(X_tr, y_tr).predict(X_te)
        acc[name] = accuracy_score(y_te, yb)
        print(
            f"{name:11s} precision {precision_score(y_te, yb, zero_division=0):.3f}  "
            f"recall {recall_score(y_te, yb, zero_division=0):.3f}  "
            f"wet-call share {yb.mean():.3f}  accuracy {acc[name]:.3f}"
        )
    print(
        f"accuracy, final minus always-dry: {acc_final - acc['always-dry']:+.3f} · "
        f"final minus nino-sign: {acc_final - acc['nino-sign']:+.3f}"
    )

    N_BOOT = 1000
    boot_rng = np.random.RandomState(RNG_SEED)
    member_rows = {
        m: np.flatnonzero(test["member_id"].values == m)
        for m in test["member_id"].unique()
    }
    names = list(member_rows)
    point = {
        "precision − base rate": precision_score(
            y_te, yhat_te, zero_division=0
        )
        - y_te.mean(),
        "AP − base rate": average_precision_score(y_te, score_te)
        - y_te.mean(),
    }
    draws = {k: [] for k in point}
    for _ in range(N_BOOT):
        picked = boot_rng.choice(names, size=len(names), replace=True)
        pick = np.concatenate([member_rows[m] for m in picked])
        yb, hb, sb = y_te[pick], yhat_te[pick], score_te[pick]
        draws["precision − base rate"].append(
            precision_score(yb, hb, zero_division=0) - yb.mean()
        )
        draws["AP − base rate"].append(
            average_precision_score(yb, sb) - yb.mean()
        )
    for k, v in draws.items():
        lo, hi = (
            round(float(b), 3) for b in np.percentile(v, [2.5, 97.5])
        )  # as printed
        reading = (
            "above no skill"
            if lo > 0
            else (
                "below no skill"
                if hi < 0
                else "NOT distinguishable from no skill"
            )
        )
        print(
            f"{reg.incumbent_name}: {k:22s} {point[k]:+.3f}   95% interval over test "
            f"members [{lo:+.3f}, {hi:+.3f}]  →  {reading}"
        )

    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    ConfusionMatrixDisplay(cm_te, display_labels=["dry", "wet"]).plot(
        ax=ax, colorbar=False, cmap=GOLD_CMAP
    )
    ax.set_xlim(-0.85, 1.85)
    ax.set_ylim(1.85, -0.85)
    ax.set_title(
        f"Test: {reg.incumbent_name} on {test.member_id.nunique()} unseen members",
        fontsize=9,
    )
    cite(fig, S.access_date)
    plt.tight_layout()

In [ ]:
if FINAL:  # reads the test labels, so it sits behind the same switch
    edges = decile_edges(train)  # the TRAINING members' edges, as in chapter 2
    test_bins, chi2_te, p_te, dof_te = wet_by_bin(test, edges)
    print(
        "wet rate by TRAINING decile of the autumn Niño 3.4 anomaly, test "
        f"members (base rate {test['wet'].mean():.3f}):"
    )
    print(test_bins.round(3).to_string())
    print(
        f"chi-square test of wet against decile, test members: "
        f"chi2 = {chi2_te:.1f}, dof = {dof_te}, p = {p_te:.2g}"
    )
    # The training shape's one repeated feature: the two extreme bins pooled
    # against the eight middle bins, on the test members.
    x = test["nino34_anom"]
    extreme = (x <= edges[1]) | (x > edges[-2])
    parts = {"two extreme bins": extreme, "eight middle bins": ~extreme}
    rates = {}
    for label, mask in parts.items():
        w = test.loc[mask, "wet"]
        rates[label] = (w.mean(), np.sqrt(w.mean() * (1 - w.mean()) / len(w)))
        print(
            f"test members, {label}: wet rate {rates[label][0]:.3f} "
            f"± {rates[label][1]:.3f} (binomial SE, n = {len(w)})"
        )
    (r_e, se_e), (r_m, se_m) = rates.values()
    print(
        f"extremes minus middle: {r_e - r_m:+.3f} ± {np.hypot(se_e, se_m):.3f} "
        "(binomial SEs treat winters as independent: untested for clustering "
        "within members)"
    )

**The shape, on the test members.** Binned with the training edges, does chapter 2's shape repeat? Compare the table row by row with
chapter 2's, then read the chi-square line: a `p` well above 0.05 means the test bins alone are not distinguishable from chance. The last
three lines pool the two extreme bins against the eight middle ones, with binomial standard errors that treat every winter as
independent — untested for clustering within members.


```{admonition} Try it: did validation predict the test?
:class: tip
Compare the test block with the register's validation scores, then with the three rung-0 lines on the same members.
```

## Justify the decision

Unidata's closing instruction: "No matter your final decision, the most important aspect is your ability to justify it using data and a
robust evaluation." 

Start from the cost chapter 4 named: a false alarm leans an outlook wet into a dry winter, the costlier error. The `false alarms per
hit` line follows from precision alone: false alarms per hit = (1 − precision) / precision, so any model with precision below 0.5 makes
more false alarms than hits, however rarely it calls wet; the wet-call share only scales the two counts. The `break-even` line turns
that into a decision: leaning wet on these calls pays only if a false alarm costs the forecaster less than the printed fraction of what a
hit is worth. An office that weighs a false alarm as heavily as a hit should leave the outlook at "equal chances". The precision edge
over the base rate is small (the interval line) and provisional (the test members were scored during development), and chapter 7's test does not replicate it.

Write the recommendation in three labelled parts, every claim pointing at a printed line:

1. **Does it deliver?** The interval, false-alarm and break-even lines.
2. **If not, say so plainly.** The answer is then the base rate, and the outlook stays at "equal chances".
3. **The next change, and its cost.** A new season, box or input is a parameter edit plus a rebuild (chapter 1); a threshold trial costs
   minutes; a snowfall target (PRECSC + PRECSL) is one more derivation.


## The result's shape

Read it from the printed lines. The straight-line relation is **indistinguishable from zero** (chapter 2's `standard errors` line;
chapter 4's coefficient). Over the training members the wet rate varies between bins of the input (chapter 2's chi-square line), the
extremes drier than the middle. On the test members the omnibus chi-square does not confirm the shape, and the single highest training
bin does not stand out again; the two extremes are again drier than the middle (the `extremes minus middle` line), untested for
clustering within members. The boosted model's precision edge is small and provisional, and chapter 7's test does not replicate it. Nor does
any of it test the NWS sign, which is stated for late winter and spring, not DJF, and for east of the Divide (limit 7).

## The limits

1. **A model-world study.** CESM2 winters are scored against CESM2 winters, never against observations. A result here is not a result for
   the real Front Range.
2. **CESM2's own ENSO and teleconnection.** The autumn state and the path that carries it to Colorado are the model's, with the model's
   biases.
3. **A ~1° grid.** It smooths the kind of high-elevation orography that Stagge et al. (2023) find amplifies the ENSO signal on the
   Intermountain West's western slopes.
4. **The season.** The shipped season is DJF; NWS Boulder names El Niño's wet signal east of the Divide "particularly during the late winter
   and spring of moderate to strong El Niños".
5. **Total precipitation, not snowfall.** The target is PRECC + PRECL, rain and snow together; a snowfall outlook is a different target.
6. **One forced-response curve for both forcing halves.** The cmip6 and smbb halves are pooled under one curve and one threshold. The
   per-half base rates in chapter 2 agree, but a half-specific forced path would be absorbed, not tested.
7. **A box that crosses the Divide.** The box's west column (centre 106.25°W, edges 106.875–105.625°W) lies mostly on or west of the
   Continental Divide, and the middle column holds the foothills (Pikes Peak, 105.04°W 38.84°N, is inside it): much of the box is
   mountain terrain, while the NWS sign it is read against is stated for east of the Divide.
8. **Not a blind test.** The test members were scored with two candidate finals during development (the top of this chapter).


## Summary

The spine was walked end to end: a question framed as a decision; data recorded from the catalogue; the forced response removed and
checked; a split by member with the leaks measured; a ladder against the base rate; trials compared by a corrected paired test, the
selection named; a provisional held-out test, read against no skill and the cost of a false alarm. Chapter 7's future test, under a protocol fixed before the run (self-attested), does not replicate the edge. The figure maps each step to its chapter.


In [ ]:
SPINE_WALKED = [
    (
        "framing",
        "chapter 1",
        "a target that does not answer the decision anyone will make",
    ),
    (
        "data",
        "chapters 1 · 2",
        "inputs whose origin and quality were never recorded",
    ),
    (
        "split",
        "chapter 3",
        "training and scoring sets that share ensemble members, so the "
        '"unseen" score is inflated',
    ),
    (
        "train",
        "chapter 4",
        "a fit that cannot be rerun because its settings and seed were "
        "not recorded",
    ),
    (
        "validate",
        "chapter 4",
        "tuning with no held-out score to show whether the model is "
        "memorising",
    ),
    ("refine", "chapters 4 · 5", "stopping at the first configuration tried"),
    (
        "test",
        "chapter 6",
        "a final number that was tuned toward through repeated use of the "
        "same held-out cases",
    ),
    (
        "justify",
        "chapter 6",
        "a model chosen for its sophistication rather than its measured "
        "performance and cost",
    ),
]
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.set(xlim=(0, len(SPINE_WALKED)), ylim=(0.22, 1))
ax.axis("off")
for k, (step, where, failure) in enumerate(SPINE_WALKED):
    ax.add_patch(
        plt.Rectangle(
            (k + 0.06, 0.6),
            0.88,
            0.32,
            facecolor="#E7CD82",
            edgecolor="#8F7022",
            lw=1.2,
        )
    )
    ax.text(
        k + 0.5,
        0.82,
        step,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
    )
    ax.add_patch(
        plt.Rectangle(
            (k + 0.12, 0.62),
            0.76,
            0.12,
            facecolor="#F5F3ED",
            edgecolor="#E4E0D6",
            lw=0.8,
        )
    )
    ax.text(k + 0.5, 0.68, where, ha="center", va="center", fontsize=8)
    ax.text(
        k + 0.5,
        0.54,
        textwrap.fill("guards against " + failure, 21),
        ha="center",
        va="top",
        fontsize=6.6,
        linespacing=1.25,
    )
    if k < len(SPINE_WALKED) - 1:
        ax.annotate(
            "",
            xy=(k + 1.06, 0.76),
            xytext=(k + 0.94, 0.76),
            arrowprops=dict(arrowstyle="->", color="#8F7022", lw=1.3),
        )
ax.set_title(
    "The supervised-learning spine as this cookbook walked it — each step, the "
    "chapter that took it, and the failure it guards against",
    fontsize=9,
)
credit_own(
    fig,
    "a diagram of the analysis spine (shape: NSF Unidata, Apache-2.0); "
    "no CESM2 LENS data drawn",
)
plt.tight_layout()

### What's next?

[Chapter 7](07-out-of-period-test.ipynb) runs the test this chapter cannot: the same members' SSP3-7.0 winters, under a protocol fixed before the scored run (self-attested; no external timestamp). Within Project Pythia, the [scikit-learn Argo](https://projectpythia.org/sklearn-argo-cookbook/),
[xbatcher ML](https://projectpythia.org/xbatcher-ML-1-cookbook/) and [Landsat ML](https://projectpythia.org/landsat-ml-cookbook/) cookbooks
take machine learning on to ocean profiles, gridded training batches and satellite imagery. As optional further material, the separate
[ncar-ml-readiness-course](https://github.com/glenbuchanan/ncar-ml-readiness-course) repository keeps the same data, split and spine and
replaces the ladder with a small neural network; nothing in it is needed here.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| Grover, M., & Banihirwe, A. (2021). *NCAR CESM2 Large Ensemble Cloud-Optimized Dataset*. NSF NCAR CISL. https://doi.org/10.26024/y48t-q717 | the dataset (required citation) | UCAR Data Terms of Use |
| Rodgers, K. B. et al. (2021). *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | the ensemble paper (required citation) | CC BY 4.0 |
| Stagge, J. H. et al. (2023). Orographic amplification of El Niño teleconnections on winter precipitation across the Intermountain West. *Nature Water* 1, 1016–1026. https://doi.org/10.1038/s44221-023-00163-9 | limit 3; the dipole and its transition zone | CC BY 4.0 |
| NWS Boulder (2020). La Niña and the Upcoming 2020-2021 Winter Season. https://www.weather.gov/media/bou/LaNinaWinterOutlook_Sept20.pdf | the sign and its season; limit 4 | US Government |
| NSF Unidata, *Supervised Machine Learning Readiness* (NSF award #2319979). https://github.com/Unidata/supervised-ml-readiness | the spine; the closing instruction | Apache-2.0 |
| Project Pythia notebook template. https://foundations.projectpythia.org/appendix/template | the shape of each chapter | CC BY 4.0 |

Results based on the CESM2 Large Ensemble acknowledge the CESM2 Large Ensemble Community Project and supercomputing resources provided by
the IBS Center for Climate Physics in South Korea. *This cookbook was self-funded and has not been reviewed or sponsored by NSF NCAR or NSF
Unidata.*
